In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Execută circuite dinamice

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



<Accordion>
<AccordionItem title="Versiuni de pachete">

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm să folosești aceste versiuni sau mai noi.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Circuitele dinamice sunt instrumente puternice cu care poți măsura qubiții în mijlocul execuției unui circuit cuantic și apoi efectua operații de logică clasică în cadrul circuitului, pe baza rezultatelor acelor măsurători mid-circuit. Acest proces este cunoscut și sub denumirea de _feedforward clasic_. Deși suntem în stadii incipiente de înțelegere a modului cel mai bun de a valorifica circuitele dinamice, comunitatea de cercetare cuantică a identificat deja o serie de cazuri de utilizare, cum ar fi:

* Pregătirea eficientă a stărilor cuantice, cum ar fi [starea GHZ](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339), [starea W](https://arxiv.org/abs/2403.07604) (pentru mai multe informații despre starea W, consultă și ["State preparation by shallow circuits using feed forward"](https://arxiv.org/abs/2307.14840)) și o clasă largă de [stări de tip produs matriceal](https://arxiv.org/abs/2404.16083)
* [Entanglement eficient la distanță mare](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) între qubiții de pe același cip, folosind circuite superficiale
* [Eșantionare eficientă a circuitelor de tip IQP](https://arxiv.org/pdf/2505.04705)

Aceste îmbunătățiri aduse de circuitele dinamice vin, însă, cu compromisuri. Măsurătorile mid-circuit și operațiile clasice au de obicei timpi de execuție mai lungi decât porțile cu doi qubiți, iar această creștere a timpului ar putea anula beneficiile aduse de reducerea adâncimii circuitului. Prin urmare, reducerea duratei măsurătorii mid-circuit este un domeniu de îmbunătățire pe care IBM Quantum&reg; îl urmărește prin lansarea [noii versiuni](/announcements/product-updates/2025-03-03-new-version-dynamic-circuits) a circuitelor dinamice. Pentru alte restricții la folosirea circuitelor dinamice, consultă tabelul de compatibilitate a funcționalităților [Estimator](/guides/estimator-options#options-compatibility-table) sau [Sampler](/guides/sampler-options#options-compatibility-table).

[Specificația OpenQASM 3](https://openqasm.com/language/classical.html#looping-and-branching) definește o serie de structuri de control al fluxului, dar Qiskit Runtime suportă în prezent doar instrucțiunea condițională `if`. În Qiskit SDK, aceasta corespunde metodei [if_test](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#if_test) pe [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit). Această metodă returnează un [manager de context](https://docs.python.org/3/reference/datamodel.html#with-statement-context-managers) și este de obicei folosită într-o instrucțiune `with`. Acest ghid descrie cum să folosești această instrucțiune condițională.

> **Note:** Exemplele de cod din acest ghid folosesc instrucțiunea standard de măsurare pentru măsurătorile mid-circuit. Totuși, este recomandat să folosești în schimb instrucțiunea `MidCircuitMeasure`, dacă backend-ul o suportă. Consultă [secțiunea Măsurători mid-circuit](#midcircuit) pentru detalii.
## Găsește backend-uri care suportă circuite dinamice
Pentru a găsi toate backend-urile la care contul tău are acces și care suportă circuite dinamice, rulează cod similar cu cel de mai jos. Acest exemplu presupune că ți-ai [salvat credențialele de autentificare](/guides/save-credentials). De asemenea, poți [specifica explicit credențialele](/guides/initialize-account#explicit) la inițializarea contului tău de serviciu Qiskit Runtime. Aceasta ți-ar permite, de exemplu, să vizualizezi backend-urile disponibile pe o anumită instanță sau tip de plan.

> **Note:** - Backend-urile disponibile pentru cont depind de instanța specificată în credențiale.
> - Noua versiune a circuitelor dinamice este acum disponibilă pentru toți utilizatorii pe toate backend-urile. Consultă [anunțul](/announcements/product-updates/2025-09-25-new-dynamic-circuits) pentru mai multe detalii.

In [1]:
# This cell is hidden from users.  It hides all those "...instance was not set..." warnings.
import warnings

warnings.filterwarnings("ignore", message=".*Instance was not set*")

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
dc_backends = service.backends(dynamic_circuits=True)
print(dc_backends)

[<IBMBackend('ibm_boston')>, <IBMBackend('ibm_pittsburgh')>, <IBMBackend('ibm_fez')>, <IBMBackend('ibm_marrakesh')>, <IBMBackend('ibm_kingston')>]


<span id="midcircuit"></span>
## Mid-circuit measurements

Prior to `qiskit-ibm-runtime` v0.43.0, `measure` was the only measurement instruction in Qiskit. Mid-circuit measurements, however, have different tuning requirements than _terminal_ measurements (measurements that happen at the end of a circuit). For example, you need to consider the instruction duration when tuning a mid-circuit measurement because longer instructions cause noisier circuits. You don't need to consider instruction duration for terminal measurements because there are no instructions after terminal measurements.

<Admonition type="note">
The `MidCircuitMeasure` instruction maps to the `measure_2` instruction reported in the backend's `supported_instructions`. However,  `measure_2` is not supported on all backends. Use `service.backends(filters=lambda b: "measure_2" in b.supported_instructions)` to find backends that support it.  New measurements might be added in the future, but this is not guarenteed.
</Admonition>

### `MidCircuitMeasure` method

In `qiskit-ibm-runtime` v0.43.0, the `MidCircuitMeasure` instruction was introduced. As the name suggests, it is a new measurement instruction that is optimized for mid-circuit on IBM&reg; QPUs.  While you can use `QuantumCircuit.measure` for a mid-circuit measurement, because of its design, `MidCircuitMeasure` is typically a better choice.  For example, it adds less overhead to your circuit than when using `QuantumCircuit.measure`.

In [3]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.circuit import MidCircuitMeasure

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, dynamic_circuits=True
)

circ = QuantumCircuit(2, 2)
circ.x(0)
circ.append(MidCircuitMeasure(), [0], [0])
# circ.measure([0], [0])
# circ.measure_all()
print(circ.draw(cregbundle=False))

     ┌───┐┌────────────┐
q_0: ┤ X ├┤0           ├
     └───┘│            │
q_1: ─────┤  Measure_2 ├
          │            │
c_0: ═════╡0           ╞
          └────────────┘
c_1: ═══════════════════
                        


<span id="midcircuit"></span>
## Măsurători mid-circuit
Înainte de `qiskit-ibm-runtime` v0.43.0, `measure` era singura instrucțiune de măsurare din Qiskit. Măsurătorile mid-circuit au, însă, cerințe de acordare diferite față de măsurătorile _terminale_ (măsurători care au loc la sfârșitul unui circuit). De exemplu, trebuie să iei în considerare durata instrucțiunii la acordarea unei măsurători mid-circuit, deoarece instrucțiunile mai lungi cauzează circuite mai zgomotoase. Nu trebuie să iei în considerare durata instrucțiunii pentru măsurătorile terminale, deoarece nu există instrucțiuni după măsurătorile terminale.

> **Note:** Instrucțiunea `MidCircuitMeasure` se mapează la instrucțiunea `measure_2` raportată în `supported_instructions` a backend-ului. Totuși, `measure_2` nu este suportată pe toate backend-urile. Folosește `service.backends(filters=lambda b: "measure_2" in b.supported_instructions)` pentru a găsi backend-urile care o suportă. Pot fi adăugate noi măsurători în viitor, dar acest lucru nu este garantat.

### Metoda `MidCircuitMeasure`
În `qiskit-ibm-runtime` v0.43.0 a fost introdusă instrucțiunea `MidCircuitMeasure`. Așa cum sugerează numele, este o nouă instrucțiune de măsurare optimizată pentru mid-circuit pe QPU-urile IBM&reg;. Deși poți folosi `QuantumCircuit.measure` pentru o măsurătoare mid-circuit, datorită design-ului său, `MidCircuitMeasure` este de obicei o alegere mai bună. De exemplu, adaugă mai puțin overhead circuitului tău față de cazul în care folosești `QuantumCircuit.measure`.

In [4]:
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, dynamic_circuits=True
)

# Create a dynamic circuit

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
qc = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

qc.h(q0)
qc.measure(q0, c0)
with qc.if_test((c0, 1)):
    qc.x(q0)
qc.measure(q0, c0)


# Convert to an ISA circuit for the given backend

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Generate samplers for backend targets
sampler = SamplerV2(backend)

# Submit jobs
sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

print(
    f">>> {' Job ID:':<10}  {sampler_job.job_id()} ({sampler_job.status()})"
)

>>>  Job ID:    d82864vtjchs73bnokf0 (DONE)


## Qiskit Runtime limitations

Be aware of the following constraints when running dynamic circuits in Qiskit Runtime.

- Due to the limited physical memory on control electronics, there is also a limit on the number of `if` statements and size of their operands. This limit is a function of the number of broadcasts and the number of broadcasted bits in a job (not a circuit).

   When processing an `if` condition, measurement data needs to be transferred to the control logic to make that evaluation. A broadcast is a transfer of unique classical data, and broadcasted bits is the number of classical bits being transferred. Consider the following:

   ```python
   c0 = ClassicalRegister(3)
   c1 = ClassicalRegister(5)
   ...
   with circuit.if_test((c0, 1)) ...
   with circuit.if_test((c0, 3)) ...
   with circuit.if_test((c1[2], 1)) ...
   ```
   In the previous code example, the first two `if_test` objects on `c0` are considered one broadcast because the content of `c0` did not change, and thus does not need to be re-broadcasted. The `if_test` on `c1` is a second broadcast. The first one broadcasts all three bits in `c0` and the second broadcasts just one bit, making a total of four broadcasted bits.

   Currently, if you broadcast 60 bits each time, then the job can have approximately 300 broadcasts. If you broadcast just one bit each time, however, then the job can have 2400 broadcasts.

- The operand used in an `if_test` statement must be 32 or fewer bits. Thus, if you are comparing an entire `ClassicalRegister`, the size of that `ClassicalRegister` must be 32 or fewer bits. If you are comparing just a single bit from a `ClassicalRegister`, however, that `ClassicalRegister` can be of any size (since the operand is only one bit).

   For example, the "Not valid" code block does not work because `cr` is more than 32 bits.  You can, however, use a classical register wider than 32 bits if you are testing only one bit, as shown in the "Valid" code block.

   <Tabs>
   <TabItem value="Not valid" label="Not valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr, 15)):
            ...
      ```
   </TabItem>
   <TabItem value="Valid" label="Valid">
      ```python
         cr = ClassicalRegister(50)
         qr = QuantumRegister(50)
         circuit = QuantumCircuit(qr, cr)
         ...
         circ.measure(qr, cr)
         with circ.if_test((cr[5], 1)):
            ...
      ```
   </TabItem>
   </Tabs>

- Nested conditionals are not allowed. For example, the following code block will not work because it has an `if_test` inside another `if_test`:
   <Tabs>
    <TabItem value="Not valid" label="Not valid">
        ```python
           c1 = ClassicalRegister(1, "c1")
           c2 = ClassicalRegister(2, "c2")
           ...
           with circ.if_test((c1, 1)):
            with circ.if_test(c2, 1)):
             ...
        ```
     </TabItem>
     <TabItem value="Valid" label="Valid">
        ```python
        cr = ClassicalRegister(2)
        ...
        with circuit.if_test((cr, 0b11)):
          ...
        ```
     </TabItem>
    </Tabs>

- Having `reset` or measurements inside conditionals is not supported.
- Arithmetic operations are not supported.
- See the [OpenQASM 3 feature table](/docs/guides/qasm-feature-table) to determine which OpenQASM 3 features are supported on Qiskit and Qiskit Runtime.
- When OpenQASM 3 (instead of `QuantumCircuit`) is used as the input format to pass circuits to Qiskit Runtime primitives, only instructions that can be loaded into Qiskit are supported. Classical operations, for example, are not supported because they cannot be loaded into Qiskit. See [Import an OpenQASM 3 program into Qiskit](/docs/guides/interoperate-qiskit-qasm3#import-an-openqasm-3-program-into-qiskit) for more information.
- The `for`, `while`, and `switch` instructions are not supported.

## Use dynamic circuits with Estimator

Since Estimator does not support dynamic circuits, you can use [Sampler](/docs/guides/get-started-with-sampler) and build your own measurement circuits instead.

To replicate Estimator's behavior, follow this process:

1. Group the terms of all observables into a partition.  This can be done by using the [`PauliList` API](/docs/api/qiskit/qiskit.quantum_info.PauliList#group_qubit_wise_commuting), for example.
     <Admonition type="note">
      You can use the [`BitArray`](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.primitives.BitArray#expectation_values) primitive attribute to compute the expectation values of the provided observables.
     </Admonition>
2. Execute one basis change circuit per partition (whichever basis change needs to be done for each partition). See the Measurement bases addon utility [`measurement_bases` module](https://qiskit.github.io/qiskit-addon-utils/apidocs/qiskit_addon_utils.exp_vals.html#qiskit_addon_utils.exp_vals.get_measurement_bases) for more information. For more information, see the [documentation](https://qiskit.github.io/qiskit-addon-utils/) for the Qiskit addon utilities package.
3. Add back together the results for each partition.

## Restrictions

Review any [Feature compatibility table](/docs/guides/estimator-options#options-compatibility-table) to understand restrictions when using dynamic circuits. Note that feature compatibility is not primitive-dependent.

## Next steps

<Admonition type="tip" title="Recommendations">
- Learn how to implement accurate dynamic decoupling by using [stretch](/docs/guides/stretch).
- Review the [classical feedforward and control flow](/docs/guides/classical-feedforward-and-control-flow) guide.
- Use [circuit schedule visualization](/docs/guides/qiskit-runtime-circuit-timing) to debug and optimize your dynamic circuits.
</Admonition>